<a href="https://colab.research.google.com/github/SarahkhIT/AgentsEngineeringProject/blob/main/notebooks/05_architecture_and_documentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Architecture & Documentation
**Solar Farm Agentic System: Part 5 of 5**

Covers **Rubric Deliverable 6 (Documentation & Evidence of Execution)**.

This project is split across five notebooks, one per rubric deliverable, so
each could be committed separately with its own meaningful commit message
instead of one bulk upload:

| Notebook | Rubric Deliverable | Pts |
|---|---|---|
| `01_agentic_reasoning_and_tools.ipynb` | 1 — Agentic Reasoning & Tool Use | 15 |
| `02_graph_orchestration_and_hitl.ipynb` | 2 — Graph-Based Orchestration (+ HITL half of 5) | 20 |
| `03_production_persistence_and_deployment.ipynb` | 5 — Production Readiness (persistence + cloud) | 20 |
| `04_security_guardrails_observability.ipynb` | 4 — Security, Guardrails & Observability | 20 |
| `05_architecture_and_documentation.ipynb` (this one) | 6 — Documentation & Evidence of Execution | 5 |

Deliverable 3 (Multi-Agent System & Role Specialization, 20 pts) isn't a
separate notebook — it's the Weather / Panel Analysis / Energy Prediction /
Maintenance / Review agents defined across notebooks 01 and 02, each with a
distinct responsibility, communicating through the shared `SolarState`
rather than one prompt role-playing multiple personas.

**Run order for the full system:** `01` → `02` → `03` → `04`, in the same
kernel session. Each notebook's opening cell notes what it depends on from
the ones before it.

---


# Solar Farm Agentic System: Architecture Write-Up

## What this is
An agentic system that monitors a solar farm: it checks live weather, reads panel
telemetry, forecasts energy output, flags maintenance issues, and produces a
human-reviewed report — all as a single LangGraph **graph**, not a linear script.

## Nodes, edges, and state
- **State**: `SolarState` (a `TypedDict`) is the single shared object every node
  reads and writes — weather data, panel status, forecast, retry count, the
  final report, and an `agent_trace` of every tool call made. `SecuredSolarState`
  extends it with `security_blocked` / `security_reason` for the guardrailed path.
- **Nodes**: `planner`, `weather`, `panel`, `energy`, `increment_retry`,
  `maintenance`, `aggregate`, `review`, `human_approval`, and `security` (added
  later) — each a specialized agent with one job.
- **Edges**: mostly fixed (`planner → weather → panel`), but two are
  **conditional**: `route_after_panel` sends the run to `maintenance` only if a
  panel group is actually faulty, and `route_after_energy` sends it back to
  `increment_retry` — the graph's **loop** — whenever the forecast confidence is
  below 0.6, up to 3 attempts, before it's allowed to continue to `aggregate`.

## Agents and coordination strategy
Four specialized agents (Weather, Panel Analysis, Energy Prediction, Maintenance)
each wrap a real tool call in an explicit **Thought → Action → Observation**
loop (`ToolCallingAgent`, the ReAct pattern) and log that trace into shared
state — this is the "agentic reasoning" layer, not a plain function call.
A fifth, the Review Agent, critiques the compiled report before a human ever
sees it. Coordination is **centralized**: there is no peer-to-peer handoff
between agents — the LangGraph orchestrator owns the state and decides which
agent runs next based on the state and the conditional edges above.

## Tools
`get_weather` (live Open-Meteo API call), `detect_faults` (computed panel
telemetry), `predict_energy` (computed forecast + confidence score), and
`recommend_maintenance` (turns faults into prioritized actions) — all real
computation over real or generated data, never hardcoded return values.

## Production concerns
- **Persistence**: a `SqliteSaver` checkpointer means the graph's state survives
  a process restart — demonstrated by invoking, interrupting, and resuming
  against the same `thread_id`.
- **Human-in-the-loop**: `human_approval_node` calls `interrupt()`, which
  pauses the graph until a human supplies `Command(resume=True/False)`.
- **Security**: an input guardrail (`ThreatDetectionAgent`, prompt-injection /
  jailbreak / credential-extraction detection) and an output guardrail
  (`ResponseSanitizationAgent`, PII/secret masking) are demonstrated against a
  real attack string, then wired into the graph itself later in this notebook
  via a `security` entry node.
- **Observability**: every tool call and guardrail decision is logged as a
  structured JSON event (`security_logs`) with component, status, and latency —
  not `print()` debugging.
- **Deployment**: `requirements.txt`, `Dockerfile`, and `docker-compose.yml`
  package the FastAPI backend (`api.py`) that exposes the reports/approval
  endpoints for a real cloud deployment.

---
Training program: Advanced Agentic AI Systems Engineering, SDAIA Academy
(delivered via Learning Space).

Session Dates: August 2nd, 2026 - August 6th, 2026

